#### CareAssist Data Preparation
The following notebook contains data preparation for the raw 2020-2024 AFCARS dataset. The cleaning steps are as follows:
- Select features of interest for modeling.
- Remove cases where 'ageatstart' exceeds 21, as 21 is the maximum age for extended foster care in most states.
- Remove cases where 'ageatlatrem' == 99, as this represents an invalid placeholder value. This caps ageatlatrem at 22, which is plausible for those who were 21 at the start of the FY.
- Engineer missing_dis feature, which is 1 if one of ['emotdist', 'phydis', 'mr', 'vishear'] is missing, else 0.
- Engineer lifelos_missing flag which is 1 if lifelos is missing, else 0, since this is ~5% of the dataset.
- Replace clindis with 'clindis_yes' and 'clindis_undetermined' to encode clindis as a binary feature with an undetermined indicator, rather than a three-level feature.
- Impute remaining missing boolean values with 0. We do this because dropping rows would reduce the dataset by 15% and disproportionately drop 2023 & 2024 data.
- Drop null non-boolean columns, since this does not materially alter the distributions and represents a small percentage of the data.
- Remove duplicates, where duplicates are defined as any rows containing the same value for 'stfcid' and 'year'.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# load data and make column names lowercase
DATA_DIR = Path("/Users/courtneychen/Library/CloudStorage/Box-Box/capstone/data")

dfs = []
for year in range(2020, 2025):
    df = pd.read_parquet(DATA_DIR / f"afcars_{year}.parquet")
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
del dfs
data.columns = data.columns.str.lower()
data.head()


,fy,version,state,st,repdatyr,repdatmo,fipscode,recnumbr,sex,amiakn,...,entered,exited,served,iswaiting,istpr,agedout,raceethn,race,stfcid,year
0,2020,1,37,NC,2020,9,8,R99044225246,2.0,1.0,...,1,0,1,0,0,0,3,3,NCR99044225246,2020
1,2020,1,37,NC,2020,9,8,R99044231032,1.0,0.0,...,0,0,1,0,0,0,1,1,NCR99044231032,2020
2,2020,1,37,NC,2020,9,37051,R99044231063,1.0,0.0,...,0,0,1,0,0,0,99,99,NCR99044231063,2020
3,2020,1,37,NC,2020,9,8,R99044232374,2.0,0.0,...,1,0,1,0,0,0,1,1,NCR99044232374,2020
4,2020,1,37,NC,2020,9,8,R99044238028,1.0,0.0,...,0,0,1,0,0,0,1,1,NCR99044238028,2020


#### Feature Selection

In [3]:
# select only relevant features for modeling
domain_variables = {
    'Metadata': ['fy', 'stfcid'], # year and fy are the same, so only keep one
    'Placement Instability': ['totalrem', 'numplep', 'lifelos', 'latremlos'],
    'Safety & Removal Context': ['phyabuse', 'sexabuse', 'neglect', 'daparent',
                                   'aaparent', 'dachild', 'aachild', 'abandmnt',
                                   'nocope', 'prtsjail', 'housing'],
    'Health & Special Needs': ['clindis', 'childis', 'emotdist', 'phydis', 'mr', 'vishear'],
    'Legal & Permanency': ['prtsdied', 'istpr', 'iswaiting', 'relinqsh'],
    'Age-Related': ['ageatlatrem', 'ageatstart', 'agedout'],
    'Current Placement': ['disreasn'],
    'Behvior': ['chbehprb']
}

# flatten into list
all_variables = [var for vars_list in domain_variables.values() for var in vars_list]

# filter data to contain selected features
data = data[all_variables]
data.head()

,fy,stfcid,totalrem,numplep,lifelos,latremlos,phyabuse,sexabuse,neglect,daparent,...,vishear,prtsdied,istpr,iswaiting,relinqsh,ageatlatrem,ageatstart,agedout,disreasn,chbehprb
0,2020,NCR99044225246,1.0,1.0,70.0,70.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0,0,0.0,0,0,0,99,0.0
1,2020,NCR99044231032,1.0,2.0,415.0,415.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0,0,0.0,0,0,0,99,0.0
2,2020,NCR99044231063,1.0,2.0,421.0,421.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0,0,0.0,0,0,0,99,0.0
3,2020,NCR99044232374,1.0,1.0,318.0,318.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0,0,0.0,6,6,0,99,0.0
4,2020,NCR99044238028,1.0,1.0,427.0,427.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0,0,0.0,0,0,0,99,0.0


#### Out-of-Range Filtering

In [4]:
# remove cases where age < 0 or age > 21
print("Length of data before:", len(data))
data = data[
    (data['ageatstart'] >= 0) &
    (data['ageatstart'] <= 21)
]
data = data[data['ageatlatrem'] != 99]
print("Length of data after:", len(data))

Length of data before: 2879158
Length of data after: 2734583


#### Addressing Missing Values

In [5]:
# create comprehensive missing value report
def missing_values_analysis(df, variables, title="Missing Values Analysis"):
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")

    missing_data = []
    for var in variables:
        if var in df.columns:
            total = len(df)
            missing = df[var].isna().sum()
            pct = (missing / total) * 100
            missing_data.append({
                'Variable': var,
                'Missing Count': missing,
                'Total Count': total,
                'Missing %': round(pct, 2),
                'Present Count': total - missing
            })

    missing_df = pd.DataFrame(missing_data)
    missing_df = missing_df.sort_values('Missing %', ascending=False)
    return missing_df

# overall missing values
overall_missing = missing_values_analysis(data, all_variables, "Overall Dataset")
print(overall_missing.to_string(index=False))


Overall Dataset
   Variable  Missing Count  Total Count  Missing %  Present Count
   emotdist         751155      2734583      27.47        1983428
     phydis         749990      2734583      27.43        1984593
         mr         749442      2734583      27.41        1985141
    vishear         746624      2734583      27.30        1987959
   abandmnt         210542      2734583       7.70        2524041
   prtsjail         205430      2734583       7.51        2529153
     nocope         184438      2734583       6.74        2550145
    lifelos         135369      2734583       4.95        2599214
   aaparent          87928      2734583       3.22        2646655
    clindis          87391      2734583       3.20        2647192
    housing          60664      2734583       2.22        2673919
   chbehprb          58929      2734583       2.15        2675654
    neglect          56106      2734583       2.05        2678477
    childis          45875      2734583       1.68        2

In [6]:
# add missingness indicator for disability features, then impute NAs with 0
disability_cols = ['emotdist', 'phydis', 'mr', 'vishear']
data['missing_dis'] = data[disability_cols].isna().any(axis=1).astype(int)
data[disability_cols] = data[disability_cols].fillna(0)
data.head()

,fy,stfcid,totalrem,numplep,lifelos,latremlos,phyabuse,sexabuse,neglect,daparent,...,prtsdied,istpr,iswaiting,relinqsh,ageatlatrem,ageatstart,agedout,disreasn,chbehprb,missing_dis
0,2020,NCR99044225246,1.0,1.0,70.0,70.0,0.0,0.0,1.0,1.0,...,0.0,0,0,0.0,0,0,0,99,0.0,0
1,2020,NCR99044231032,1.0,2.0,415.0,415.0,0.0,0.0,1.0,0.0,...,0.0,0,0,0.0,0,0,0,99,0.0,0
2,2020,NCR99044231063,1.0,2.0,421.0,421.0,0.0,0.0,1.0,1.0,...,0.0,0,0,0.0,0,0,0,99,0.0,0
3,2020,NCR99044232374,1.0,1.0,318.0,318.0,0.0,0.0,1.0,0.0,...,0.0,0,0,0.0,6,6,0,99,0.0,0
4,2020,NCR99044238028,1.0,1.0,427.0,427.0,0.0,0.0,1.0,1.0,...,0.0,0,0,0.0,0,0,0,99,0.0,0


In [7]:
# check binary variables for missing or nonbinary values
nonbin_fields = ['fy', 'stfcid', 'lifelos', 'latremlos', 'ageatlatrem', 'disreasn', 'totalrem', 'numplep', 'clindis']
binary_fields = [col for col in data.columns if col not in nonbin_fields]

for field in binary_fields:
    null_count = data[field].isna().sum()
    null_pct = data[field].isna().mean() * 100
    print(f"{field}: {null_count} missing ({null_pct:.2f}%)")

phyabuse: 15690 missing (0.57%)
sexabuse: 15688 missing (0.57%)
neglect: 56106 missing (2.05%)
daparent: 40180 missing (1.47%)
aaparent: 87928 missing (3.22%)
dachild: 15694 missing (0.57%)
aachild: 15688 missing (0.57%)
abandmnt: 210542 missing (7.70%)
nocope: 184438 missing (6.74%)
prtsjail: 205430 missing (7.51%)
housing: 60664 missing (2.22%)
childis: 45875 missing (1.68%)
emotdist: 0 missing (0.00%)
phydis: 0 missing (0.00%)
mr: 0 missing (0.00%)
vishear: 0 missing (0.00%)
prtsdied: 15688 missing (0.57%)
istpr: 0 missing (0.00%)
iswaiting: 0 missing (0.00%)
relinqsh: 18299 missing (0.67%)
ageatstart: 0 missing (0.00%)
agedout: 0 missing (0.00%)
chbehprb: 58929 missing (2.15%)
missing_dis: 0 missing (0.00%)


In [8]:
# missing value count
print("Original size:", len(data))

dropped_all = data.dropna()
print("After dropping any missing:", len(dropped_all))

Original size: 2734583
After dropping any missing: 2337062


In [9]:
# create flag for any missing in those columns
data['any_missing'] = data[binary_fields].isna().any(axis=1)

# count rows per year with any missing
missing_by_year = (
    data.groupby('fy')['any_missing']
      .sum()
      .sort_index()
)

# also get total rows per year for context
total_by_year = data.groupby('fy').size().sort_index()

# combine
summary = pd.DataFrame({
    'rows_with_missing': missing_by_year,
    'total_rows': total_by_year
})

summary['percent_missing_rows'] = (
    summary['rows_with_missing'] / summary['total_rows']
)

summary

,rows_with_missing,total_rows,percent_missing_rows
fy,,,
2020,2469,600254,0.004113
2021,2167,574100,0.003775
2022,2301,541386,0.004250
2023,103716,522004,0.198688
2024,105116,496839,0.211570


In [10]:
# impute remaining boolean features with 0
data.drop(columns='any_missing', inplace=True)
data[binary_fields] = data[binary_fields].fillna(0)
data.head()

,fy,stfcid,totalrem,numplep,lifelos,latremlos,phyabuse,sexabuse,neglect,daparent,...,prtsdied,istpr,iswaiting,relinqsh,ageatlatrem,ageatstart,agedout,disreasn,chbehprb,missing_dis
0,2020,NCR99044225246,1.0,1.0,70.0,70.0,0.0,0.0,1.0,1.0,...,0.0,0,0,0.0,0,0,0,99,0.0,0
1,2020,NCR99044231032,1.0,2.0,415.0,415.0,0.0,0.0,1.0,0.0,...,0.0,0,0,0.0,0,0,0,99,0.0,0
2,2020,NCR99044231063,1.0,2.0,421.0,421.0,0.0,0.0,1.0,1.0,...,0.0,0,0,0.0,0,0,0,99,0.0,0
3,2020,NCR99044232374,1.0,1.0,318.0,318.0,0.0,0.0,1.0,0.0,...,0.0,0,0,0.0,6,6,0,99,0.0,0
4,2020,NCR99044238028,1.0,1.0,427.0,427.0,0.0,0.0,1.0,1.0,...,0.0,0,0,0.0,0,0,0,99,0.0,0


In [11]:
# look at remaining fields
for field in nonbin_fields:
    null_count = data[field].isna().sum()
    null_pct = data[field].isna().mean() * 100
    print(f"{field}: {null_count} missing ({null_pct:.2f}%)")

fy: 0 missing (0.00%)
stfcid: 0 missing (0.00%)
lifelos: 135369 missing (4.95%)
latremlos: 397 missing (0.01%)
ageatlatrem: 0 missing (0.00%)
disreasn: 0 missing (0.00%)
totalrem: 2878 missing (0.11%)
numplep: 10359 missing (0.38%)
clindis: 87391 missing (3.20%)


Check if removing null values in remaining fields affects age, removals, days in foster care, or outcome distributions. This is to ensure we don't introduce biases during dropping. 

In [12]:
cols_to_check = ['latremlos', 'totalrem', 'numplep', 'clindis']

# map adoption outcomes with success/other
success_map = {
    1: 1,      # Reunified
    2: 1,      # Living with relatives
    3: 1,      # Adoption
    4: 0,        # Emancipation
    5: 1,      # Guardianship
    6: 0,        # Transfer
    7: 0,        # Runaway
    8: 0,     # Death
}

data["discharge_group"] = data["disreasn"].map(success_map)

results = []

for col in cols_to_check:
    
    temp = data.copy()
    temp['is_missing'] = temp[col].isna()
    
    summary = temp.groupby('is_missing').agg(
        count=('stfcid', 'count'),
        pct_of_dataset=('stfcid', lambda x: len(x) / len(data)),
        avg_age=('ageatlatrem', 'mean'),
        avg_totalrem=('totalrem', 'mean'),
        avg_lifelos=('lifelos', 'mean'),
        avg_outcome=('discharge_group', 'mean')
        
    ).reset_index()
    
    summary['variable'] = col
    results.append(summary)

final_summary = pd.concat(results)

final_summary = final_summary[
    ['variable','is_missing','count','pct_of_dataset',
     'avg_outcome', 'avg_age','avg_totalrem','avg_lifelos']
]

final_summary.sort_values(['variable','is_missing'])

,variable,is_missing,count,pct_of_dataset,avg_outcome,avg_age,avg_totalrem,avg_lifelos
0,clindis,False,2647192,0.968042,0.891581,6.896096,1.269790,788.607658
1,clindis,True,87391,0.031958,0.923889,7.260519,1.166014,545.042517
0,latremlos,False,2734186,0.999855,0.892981,6.906450,1.266344,780.798848
1,latremlos,True,397,0.000145,0.221024,15.801008,2.177546,NaN
0,numplep,False,2724224,0.996212,0.892618,6.904259,1.265782,781.640138
1,numplep,True,10359,0.003788,0.911779,7.823535,1.450890,518.658208
0,totalrem,False,2731705,0.998948,0.892934,6.906181,1.266471,780.798848
1,totalrem,True,2878,0.001052,0.757353,8.389159,NaN,NaN


In [13]:
# add lifelos_missing flag since missingnes is structural and impute with 0
data['lifelos_missing'] = data['lifelos'].isna().astype(int)
data['lifelos'] = data['lifelos'].fillna(0)

# drop remaining null values since missingness is random and a small percentage of the dataset
print("Length data before drop:", len(data))
data = data.dropna(subset=nonbin_fields)
print("Length of data after drop:", len(data))

print("Count of remaining NA fields", data.isna().sum().sum())

Length data before drop: 2734583
Length of data after drop: 2636498
Count of remaining NA fields 1676133


Since clindis includes a “Not Yet Determined” level, we create two indicator variables:
- clindis_yes (1 if confirmed disability, 0 otherwise), and
- clindis_undetermined (1 if disability status is not yet determined, 0 otherwise).

In [28]:
# since clindis is 
data['clindis_yes'] = (data['clindis'] == 1).astype(int)
data['clindis_undetermined'] = (data['clindis'] == 3).astype(int)

data = data.drop(columns='clindis')

####  Removing Duplicates

In [29]:
# remove duplicates
prev_len = len(data)

data = (
    data
    .drop_duplicates(subset=['stfcid', 'fy'], keep='first')
    .reset_index(drop=True)
)

curr_len = len(data)
print(f"Removed {prev_len - curr_len} duplicate rows")

Removed 48 duplicate rows
